In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_person AS
SELECT * FROM VALUES
    (1001, 8507, 1980),
    (1002, 8532, 1975)
AS tmp_person (
    person_id,
    gender_concept_id,
    year_of_birth
);


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_visit_occurrence AS
SELECT * FROM VALUES
    (2001, 1001, 9202, DATE('2025-01-10'), TIMESTAMP('2025-01-10 09:00:00')),
    (2002, 1001, 9202, DATE('2025-03-15'), TIMESTAMP('2025-03-15 08:30:00')),
    (2003, 1002, 9202, DATE('2025-02-20'), TIMESTAMP('2025-02-20 14:00:00'))
AS tmp_visit_occurrence (
    visit_occurrence_id,
    person_id,
    visit_concept_id,
    visit_start_date,
    visit_start_datetime
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_condition_occurrence AS
SELECT * FROM VALUES
    (3001, 1001, 201826, DATE('2025-01-10'), TIMESTAMP('2025-01-10 09:10:00'), 2001, 'Type 2 diabetes mellitus'),
    (3002, 1001, 201826, DATE('2025-03-15'), TIMESTAMP('2025-03-15 08:40:00'), 2002, 'Type 2 diabetes mellitus'),
    (3003, 1002, 4329847, DATE('2025-02-20'), TIMESTAMP('2025-02-20 14:10:00'), 2003, 'Hypertension')
AS tmp_condition_occurrence (
    condition_occurrence_id,
    person_id,
    condition_concept_id,
    condition_start_date,
    condition_start_datetime,
    visit_occurrence_id,
    condition_source_value
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_drug_exposure AS
SELECT * FROM VALUES
    (5001, 1001, 1503297, DATE('2025-01-10'), DATE('2025-02-09'), 2001, 'Metformin 500 MG'),
    (5002, 1001, 1503297, DATE('2025-03-15'), DATE('2025-04-14'), 2002, 'Metformin 500 MG'),
    (5003, 1002, 1308216, DATE('2025-02-20'), DATE('2025-03-21'), 2003, 'Lisinopril 10 MG')
AS tmp_drug_exposure (
    drug_exposure_id,
    person_id,
    drug_concept_id,
    drug_exposure_start_date,
    drug_exposure_end_date,
    visit_occurrence_id,
    drug_source_value
);

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_procedure_occurrence AS
SELECT * FROM VALUES
    (6001, 1001, 4150814, DATE('2025-01-10'), TIMESTAMP('2025-01-10 09:40:00'), 2001, 'Diabetes education'),
    (6002, 1001, 4150814, DATE('2025-03-15'), TIMESTAMP('2025-03-15 09:00:00'), 2002, 'Diabetes education follow-up')
AS tmp_procedure_occurrence (
    procedure_occurrence_id,
    person_id,
    procedure_concept_id,
    procedure_date,
    procedure_datetime,
    visit_occurrence_id,
    procedure_source_value
);


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fr_condition_visit AS
SELECT
    19 AS domain_concept_id_1, -- Condition
    condition_occurrence.condition_occurrence_id AS fact_id_1,
    8 AS domain_concept_id_2,  -- Visit
    visit_occurrence.visit_occurrence_id AS fact_id_2,
    33136 AS relationship_concept_id -- Has associated visit (OMOP)
FROM tmp_condition_occurrence AS condition_occurrence
JOIN tmp_visit_occurrence AS visit_occurrence
    ON visit_occurrence.visit_occurrence_id = condition_occurrence.visit_occurrence_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fr_measurement_visit AS
SELECT
    21 AS domain_concept_id_1, -- Measurement
    measurement.measurement_id AS fact_id_1,
    8 AS domain_concept_id_2,  -- Visit
    visit_occurrence.visit_occurrence_id AS fact_id_2,
    33136 AS relationship_concept_id -- Has associated visit (OMOP)
FROM tmp_measurement AS measurement
JOIN tmp_visit_occurrence AS visit_occurrence
    ON visit_occurrence.visit_occurrence_id = measurement.visit_occurrence_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fr_drug_visit AS
SELECT
    13 AS domain_concept_id_1, -- Drug
    drug_exposure.drug_exposure_id AS fact_id_1,
    8 AS domain_concept_id_2,  -- Visit
    visit_occurrence.visit_occurrence_id AS fact_id_2,
    33136 AS relationship_concept_id -- Has associated visit (OMOP)
FROM tmp_drug_exposure AS drug_exposure
JOIN tmp_visit_occurrence AS visit_occurrence
    ON visit_occurrence.visit_occurrence_id = drug_exposure.visit_occurrence_id;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fr_measurement_condition AS
SELECT
    21 AS domain_concept_id_1, -- Measurement
    measurement.measurement_id AS fact_id_1,
    19 AS domain_concept_id_2, -- Condition
    condition_occurrence.condition_occurrence_id AS fact_id_2,
    44818770 AS relationship_concept_id -- Has associated finding (SNOMED)
FROM tmp_measurement AS measurement
JOIN tmp_condition_occurrence AS condition_occurrence
    ON condition_occurrence.person_id = measurement.person_id
   AND condition_occurrence.visit_occurrence_id = measurement.visit_occurrence_id;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fr_procedure_visit AS
SELECT
    10 AS domain_concept_id_1, -- Procedure
    procedure_occurrence.procedure_occurrence_id AS fact_id_1,
    8 AS domain_concept_id_2,  -- Visit
    visit_occurrence.visit_occurrence_id AS fact_id_2,
    33136 AS relationship_concept_id -- Has associated visit (OMOP)
FROM tmp_procedure_occurrence AS procedure_occurrence
JOIN tmp_visit_occurrence AS visit_occurrence
    ON visit_occurrence.visit_occurrence_id = procedure_occurrence.visit_occurrence_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fr_condition_procedure AS
SELECT
    19 AS domain_concept_id_1, -- Condition
    condition_occurrence.condition_occurrence_id AS fact_id_1,
    10 AS domain_concept_id_2, -- Procedure
    procedure_occurrence.procedure_occurrence_id AS fact_id_2,
    44818784 AS relationship_concept_id -- Has associated procedure (SNOMED)
FROM tmp_condition_occurrence AS condition_occurrence
JOIN tmp_procedure_occurrence AS procedure_occurrence
    ON procedure_occurrence.person_id = condition_occurrence.person_id
   AND procedure_occurrence.visit_occurrence_id = condition_occurrence.visit_occurrence_id;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tmp_fact_relationship AS
SELECT * FROM fr_condition_visit
UNION ALL
SELECT * FROM fr_measurement_visit
UNION ALL
SELECT * FROM fr_drug_visit
UNION ALL
SELECT * FROM fr_measurement_condition
UNION ALL
SELECT * FROM fr_procedure_visit
UNION ALL
SELECT * FROM fr_condition_procedure;


In [0]:
%sql
SELECT *
FROM tmp_fact_relationship
ORDER BY domain_concept_id_1, fact_id_1, domain_concept_id_2, fact_id_2;
